# ClarityPath — LoRA Fine-Tuning

Before running: set runtime to **T4 GPU** (Runtime > Change runtime type)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install deps

In [ ]:
!pip install -q diffusers transformers torch accelerate peft safetensors Pillow

## 3. Clone repo

In [ ]:
!git clone -b Bilal927-Branch https://github.com/bilalahmed1905/calgaryhacks2026.git
%cd calgaryhacks2026

## 4. Upload training images

Click **Choose Files** and upload your images (PNG/JPG).

In [ ]:
import os
from google.colab import files

os.makedirs("training_images", exist_ok=True)

uploaded = files.upload()
for filename, data in uploaded.items():
    with open(os.path.join("training_images", filename), "wb") as f:
        f.write(data)

print(f"Uploaded {len(uploaded)} images")

## 5. HuggingFace token

Get one from https://huggingface.co/settings/tokens

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HF token: ")

## 6. Train

In [ ]:
!python finetune.py ./training_images "university student career development, professional workspace, authentic, motivational"

## 7. Test it

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from peft import PeftModel

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    token=os.environ.get("HF_TOKEN"),
).to("cuda")

pipe.unet = PeftModel.from_pretrained(
    pipe.unet,
    "outputs/lora_weights/claritypath_lora",
)

# Test with a Module 1 style prompt
image = pipe(
    "university student confidently presenting their unique skills to employers, software engineering, warm lighting, professional photography",
    num_inference_steps=30,
).images[0]

image

## 8. Download weights

Save these before the session ends.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("claritypath_lora", "zip", "outputs/lora_weights/claritypath_lora")
files.download("claritypath_lora.zip")